# Applied RAG with My Own Data — Wine Portfolio Assistant

This notebook implements the **Retrieval Augmented Generation (RAG)** pattern over my own
real wine portfolio (a Nexus stock export) instead of the example wine-review data.

**Pipeline**
1. Load the portfolio with Pandas from a CSV file
2. Format it as a list of dictionaries
3. Create embeddings with Sentence Transformers
4. Store and index the vectors in an in-memory Qdrant database
5. Retrieve the most relevant wines for a question (semantic search)
6. Augment a prompt with the retrieved context and generate an answer with an LLM
   served locally by **Llamafile** (OpenAI-compatible endpoint).

## 0. Install dependencies

Run once. Skip if your environment already has them.

In [ ]:
%pip install -q sentence-transformers qdrant-client pandas openai

## 1. Load the data with Pandas

`wine_portfolio.csv` holds each wine with its producer, type, region, vintage, stock,
storage, ROI, and a free-text `description` we will embed.

In [ ]:
import pandas as pd

df = pd.read_csv("data/wine_portfolio.csv")
print(f"Loaded {len(df)} wines")
df.head()

## 2. Format the data as a list of dictionaries

Qdrant stores a *payload* (arbitrary JSON) next to each vector. A list of dicts is the easiest
structure to ingest — every dict becomes one record's payload.

In [ ]:
data = df.to_dict(orient="records")
data[0]

## 3. Create the embeddings encoder

`all-MiniLM-L6-v2` is small, fast, and produces 384-dimensional sentence embeddings — a great
default for RAG. The first run downloads the model (~80 MB).

In [ ]:
from sentence_transformers import SentenceTransformer

encoder = SentenceTransformer("all-MiniLM-L6-v2")
dim = encoder.get_sentence_embedding_dimension()
print(f"Embedding dimension: {dim}")

## 4. Create the Qdrant collection

We run Qdrant fully **in-memory** (`:memory:`) — nothing to install or host. The vector size
must match the encoder, and we use cosine distance for semantic similarity.

In [ ]:
from qdrant_client import QdrantClient
from qdrant_client.models import VectorParams, Distance

client = QdrantClient(":memory:")

COLLECTION = "wine_portfolio"
client.recreate_collection(
    collection_name=COLLECTION,
    vectors_config=VectorParams(size=dim, distance=Distance.COSINE),
)

## 5. Embed and upload the data

Each wine's `description` is encoded into a vector. The full dictionary is kept as the payload
so all the structured fields come back at query time.

In [ ]:
from qdrant_client.models import PointStruct

points = [
    PointStruct(
        id=idx,
        vector=encoder.encode(doc["description"]).tolist(),
        payload=doc,
    )
    for idx, doc in enumerate(data)
]

client.upload_points(collection_name=COLLECTION, points=points)
print(f"Uploaded {len(points)} vectors to the '{COLLECTION}' collection")

## 6. Retrieve — semantic search

This is the **Retrieval** in RAG. The query is embedded with the *same* encoder and Qdrant
returns the nearest wines — even when the query shares few exact keywords with the stored text.

In [ ]:
def search(query, limit=4):
    hits = client.query_points(
        collection_name=COLLECTION,
        query=encoder.encode(query).tolist(),
        limit=limit,
    ).points
    return hits

query = "Italian red wines from Tuscany"
for hit in search(query):
    p = hit.payload
    print(f"{hit.score:.3f}  {p['name']} {p['vintage']}  ({p['type']})")

## 7. Connect to the LLM (Llamafile)

Download a Llamafile (the Phi-2 model, ~2 GB, works well) and start it before running this:

```bash
chmod +x phi-2.Q4_K_M.llamafile
./phi-2.Q4_K_M.llamafile --server --nobrowser
```

Llamafile speaks the OpenAI API, so we use the official `openai` client and point `base_url`
at localhost. No API key is required.

In [ ]:
from openai import OpenAI

llm = OpenAI(
    base_url="http://localhost:8080/v1",  # Llamafile's OpenAI-compatible endpoint
    api_key="sk-no-key-required",          # Llamafile ignores the key
)

## 8. Augment + Generate — the full RAG query

We retrieve the most relevant wines, inject them as **context**, and ask the LLM to answer using
*only* that context. This grounds the model in my own portfolio and reduces hallucination.

In [ ]:
import json

SYSTEM_PROMPT = (
    "You are a knowledgeable wine cellar assistant for a private portfolio. "
    "Answer the user's question using ONLY the wine context provided. "
    "Refer to specific wines by name and vintage and briefly justify each pick. "
    "If the context has no good match, say so honestly."
)

def rag_query(question, limit=4):
    hits = search(question, limit=limit)
    context = [hit.payload for hit in hits]
    user_message = (
        f"Question: {question}\n\n"
        f"Wine context (JSON):\n{json.dumps(context, indent=2)}"
    )
    completion = llm.chat.completions.create(
        model="LLaMA_CPP",
        messages=[
            {"role": "system", "content": SYSTEM_PROMPT},
            {"role": "user", "content": user_message},
        ],
        temperature=0.4,
    )
    return completion.choices[0].message.content, context

answer, used = rag_query("Which Italian reds do I have, and are any ready to drink?")
print(answer)
print("\n--- retrieved ---")
for p in used:
    print("-", p["name"], p["vintage"])

## 9. Try your own questions

Because retrieval is semantic, natural questions work even without exact keyword matches.

In [ ]:
questions = [
    "What are my best wines by ROI?",
    "Which white wines do I have in stock?",
    "What is stored in bond at Nexus?",
]

for q in questions:
    answer, _ = rag_query(q)
    print(f"Q: {q}\nA: {answer}\n{'='*70}")

## Done!

A working RAG pipeline over my own wine portfolio:
**Pandas → list of dicts → Sentence Transformers → Qdrant retrieval → LLM generation.**
Swap `data/wine_portfolio.csv` for any CSV (keep a text column to embed) to repoint it.